In [ ]:
# ============================================================
#  Module 2 — Setup & Why LLMs Need Embeddings
#  DATA 1010 – Artificial Intelligence in Action
# ============================================================

# This cell:
#   • Installs and loads a free, open-source embedding model
#   • Asks for a group code (like Labs 1–4)
#   • Generates a test embedding for a sample sentence
#   • Shows the embedding dimensionality
#   • Confirms the Colab environment is working

# -----------------------------
# 1. Install requirements
# -----------------------------
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    !pip install -q sentence-transformers
    from sentence_transformers import SentenceTransformer

# -----------------------------
# 2. Group Code
# -----------------------------
group_code = input("Enter your group code (an integer): ")
print(f"Group code set to: {group_code}")

# -----------------------------
# 3. Load the embedding model
# -----------------------------
print("\nLoading embedding model (all-MiniLM-L6-v2)...")
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("Model loaded successfully!")

# -----------------------------
# 4. Generate a test embedding
# -----------------------------
test_sentence = "Astronomy is the study of stars and galaxies."

print("\nGenerating a test embedding...")
embedding = model.encode(test_sentence)

print("\nSample sentence:")
print("   ", test_sentence)
print("\nEmbedding generated!")
print(f"Embedding vector length: {len(embedding)} dimensions")
print(f"First 10 values: {embedding[:10]}")

# -----------------------------
# Finished
# -----------------------------
print("\nSetup complete. You are ready for Module 2.")


In [ ]:
# ============================================================
#  Module 2 — Activity 1: Word & Sentence Embeddings
#  DATA 1010 – Artificial Intelligence in Action
# ============================================================

# This cell:
#   • Defines a small set of sentences
#   • Uses the embedding model from Module 0 to encode them
#   • Reduces embeddings to 2D using PCA
#   • Plots the sentences as points in a 2D "meaning space"



# -----------------------------
# 1. Define the sentences
# -----------------------------
sentences = [
    "The cat sat on the mat.",
    "Cats are great pets.",
    "Stars fuse hydrogen into helium.",
    "Galaxies contain billions of stars.",
    "Neural networks learn patterns."
]

print("Sentences to embed:\n")
for i, s in enumerate(sentences, start=1):
    print(f"{i}. {s}")
print("\nEncoding sentences into embeddings...")

# -----------------------------
# 2. Encode sentences
# -----------------------------
# Uses the SentenceTransformer model loaded in Module 0
embeddings = model.encode(sentences)
embeddings = np.array(embeddings)

print("Done!")
print(f"Embedding array shape: {embeddings.shape}  (sentences x dimensions)")

# -----------------------------
# 3. Reduce to 2D with PCA
# -----------------------------
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

print("\nPCA complete. Showing 2D coordinates for each sentence:")
for i, (x, y) in enumerate(embeddings_2d, start=1):
    print(f"{i}. ({x:.3f}, {y:.3f})")

# -----------------------------
# 4. Plot the 2D embedding space
# -----------------------------
plt.figure(figsize=(6, 6))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1])

for i, (x, y) in enumerate(embeddings_2d):
    label = f"{i}"
    plt.text(x + 0.01, y + 0.01, label)

plt.title("2D PCA Projection of Sentence Embeddings")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.axhline(0, linewidth=0.5)
plt.axvline(0, linewidth=0.5)
plt.grid(True)
plt.show()

print("\nEach point is one sentence. The numbers on the plot match the list above.")
print("Discuss with your group:")
print("  • Which sentences are closest together?")
print("  • Do the clusters match your intuition about meaning?")
print("  • Why might 'Neural networks learn patterns.' sit apart from the others?")


In [ ]:
# ============================================================
#  Module 2 — Activity 2: Cosine Similarity Exploration
#  DATA 1010 – Artificial Intelligence in Action
# ============================================================

# This cell:
#   • Lets you choose any two sentences from Activity 1
#   • Computes cosine similarity + distance
#   • Highlights the two chosen points on the PCA plot
#   • Helps you compare intuition vs measurable similarity

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets

# ---------------------------------------
# 1. Dropdown menus for sentence selection
# ---------------------------------------
sentence_options = [f"{i+1}. {sentences[i]}" for i in range(len(sentences))]

dropdown1 = widgets.Dropdown(
    options=sentence_options,
    description='Sentence 1:',
    style={'description_width': 'initial'},
    value=sentence_options[0]
)

dropdown2 = widgets.Dropdown(
    options=sentence_options,
    description='Sentence 2:',
    style={'description_width': 'initial'},
    value=sentence_options[1]
)

display(dropdown1)
display(dropdown2)

# ---------------------------------------
# 2. Button to run the comparison
# ---------------------------------------
button = widgets.Button(description="Compute Similarity", button_style='info')
output = widgets.Output()
display(button, output)

# ---------------------------------------
# 3. Callback function for button press
# ---------------------------------------
def on_button_clicked(b):
    with output:
        output.clear_output()

        # Extract indices
        idx1 = int(dropdown1.value.split(".")[0]) - 1
        idx2 = int(dropdown2.value.split(".")[0]) - 1
        
        emb1 = embeddings[idx1].reshape(1, -1)
        emb2 = embeddings[idx2].reshape(1, -1)
        
        sim = cosine_similarity(emb1, emb2)[0][0]
        dist = 1 - sim
        
        print("===============================================")
        print("Sentence 1:")
        print("  ", sentences[idx1], "\n")
        print("Sentence 2:")
        print("  ", sentences[idx2])
        print("===============================================\n")
        
        print(f"Cosine Similarity: {sim:.4f}")
        print(f"Cosine Distance:   {dist:.4f}")
        
        # ---------------------------------------
        # Visualize the two points on the PCA scatterplot
        # ---------------------------------------
        plt.figure(figsize=(6, 6))
        plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], color='gray', alpha=0.6)
        
        # Highlight selected points
        plt.scatter(embeddings_2d[idx1, 0], embeddings_2d[idx1, 1], color='red', s=120, label="Sentence 1")
        plt.scatter(embeddings_2d[idx2, 0], embeddings_2d[idx2, 1], color='blue', s=120, label="Sentence 2")
        
        # Labels for all points
        for i, (x, y) in enumerate(embeddings_2d):
            plt.text(x + 0.01, y + 0.01, str(i+1))
        
        plt.title("PCA Visualization of Selected Sentences")
        plt.xlabel("PCA Component 1")
        plt.ylabel("PCA Component 2")
        plt.legend()
        plt.grid(True)
        plt.show()

button.on_click(on_button_clicked)

In [ ]:
#@title ### 🔎 Module 3 — Semantic Search Mini-Application
#@markdown
# # Module 3 — Semantic Search Mini-Application
# In this activity, you'll build a tiny **semantic search engine** using the same
# embedding model from earlier in the lab.
#
# Instead of searching by keywords, you'll search by **meaning**:
#
# 1. Embed a small corpus of mixed-topic sentences.
# 2. Embed a **user query** (in natural language).
# 3. Compute cosine similarities between the query and all corpus sentences.
# 4. Show the **top 3 most similar** sentences.
# 5. Visualize the query in the same 2D PCA space as the corpus.
#
# Try queries like:
# - "objects that orbit the sun"
# - "examples of nuclear fusion"
# - "animals people keep at home"
# - "how machines learn patterns"

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------------------------
# 1. Define a small mixed-topic corpus
# ------------------------------------------------
corpus = [
    # Astronomy-related
    "The Earth orbits the Sun once every year.",
    "Stars fuse hydrogen into helium in their cores.",
    "Galaxies contain billions of stars.",
    "The Moon causes tides in the oceans.",
    "Telescopes help astronomers observe distant galaxies.",

    # Pets / animals
    "Cats are popular pets that like to nap.",
    "Dogs are loyal animals that enjoy walks.",
    "Many people keep fish in aquariums at home.",
    "Birds can be trained to mimic human speech.",
    "Hamsters often run on wheels in their cages.",

    # Machine learning / AI
    "Neural networks learn patterns from data.",
    "Machine learning models can recognize images.",
    "Large language models generate human-like text.",
    "Training a model requires lots of examples.",
    "Embeddings represent meaning as high-dimensional vectors.",

    # Miscellaneous
    "Cooking at home can be fun and relaxing.",
    "Music concerts bring people together.",
    "Running is a good form of exercise.",
    "Libraries are quiet places to read and study.",
    "Video games can be played with friends online."
]

print("Our corpus has", len(corpus), "sentences.\n")

# ------------------------------------------------
# 2. Embed the corpus
# ------------------------------------------------
print("Encoding corpus sentences into embeddings...")
corpus_embeddings = model.encode(corpus)
corpus_embeddings = np.array(corpus_embeddings)
print("Done. Shape:", corpus_embeddings.shape, "(sentences x dimensions)")

# ------------------------------------------------
# 3. Fit PCA on the corpus for 2D visualization
# ------------------------------------------------
pca_corpus = PCA(n_components=2)
corpus_2d = pca_corpus.fit_transform(corpus_embeddings)

# ------------------------------------------------
# 4. Get a user query and embed it
# ------------------------------------------------
print("\nEnter a natural-language query to search the corpus by meaning.")
print("Example queries:")
print("  • objects that orbit the sun")
print("  • examples of nuclear fusion")
print("  • animals people keep at home")
print("  • how machines learn patterns\n")

query = input("Type your query here: ").strip()
if not query:
    query = "objects that orbit the sun"
    print(f"(No input detected. Using default query: '{query}')")

print("\nEmbedding your query...")
query_embedding = model.encode([query])
query_2d = pca_corpus.transform(query_embedding)  # project into same PCA space

# ------------------------------------------------
# 5. Compute cosine similarity and get top 3 results
# ------------------------------------------------
sims = cosine_similarity(query_embedding, corpus_embeddings)[0]  # shape: (N,)
top_indices = np.argsort(sims)[::-1][:3]  # indices of top 3

print("\n================ SEMANTIC SEARCH RESULTS ================\n")
print("Your query:")
print("  ", query, "\n")
print("Top 3 most similar sentences in the corpus:\n")

for rank, idx in enumerate(top_indices, start=1):
    print(f"{rank}. [{idx}]  (similarity = {sims[idx]:.4f})")
    print("   ", corpus[idx], "\n")

# ------------------------------------------------
# 6. Visualize corpus + query in PCA space
# ------------------------------------------------
plt.figure(figsize=(7, 7))

# Plot corpus points
plt.scatter(corpus_2d[:, 0], corpus_2d[:, 1], alpha=0.6, label="Corpus sentences")

# Highlight top 3
plt.scatter(corpus_2d[top_indices, 0], corpus_2d[top_indices, 1],
            color="orange", s=120, label="Top 3 matches")

# Plot query
plt.scatter(query_2d[0, 0], query_2d[0, 1], color="red", s=160, label="Query")

# Label a few points (indices)
for i, (x, y) in enumerate(corpus_2d):
    plt.text(x + 0.01, y + 0.01, str(i), fontsize=8)

plt.title("Semantic Search: Corpus + Query in 2D PCA Space")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.axhline(0, linewidth=0.5)
plt.axvline(0, linewidth=0.5)
plt.grid(True)
plt.legend()
plt.show()

print("\nEach point is a sentence from the corpus.")
print("The red point is your query; orange points are the top 3 matches.")
print("Discuss with your group: Do the results make sense for your query?")
